# 📊 DVF Data Exploration — Step 1
## Real Estate Analysis in Paris

**Author:** Ahmed Maala  
**Team:** Ahmed Maala & Natalja Voth  
**Date:** May 9, 2026  
**Project:** Liora Data Engineering Training

---

## 🎯 Objectives
1. Download DVF dataset from data.gouv.fr
2. Explore the data structure
3. Identify missing values
4. Filter data for Paris only
5. Save cleaned dataset

In [ ]:
# ===== Standard Libraries =====
import os
import sys
from pathlib import Path

# ===== Data Manipulation =====
import pandas as pd
import numpy as np

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Web Requests =====
import requests

# ===== Display Settings =====
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# ===== Plot Style =====
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

# ===== Confirmation =====
print("✅ All libraries imported successfully!")
print(f"📦 pandas version: {pd.__version__}")
print(f"📦 numpy version: {np.__version__}")

## 📥 Step 1: Download DVF Dataset

We download the 2022 DVF dataset from the official French government 
open data portal (data.gouv.fr).

**Source:** https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/

**File:** Compressed CSV (~150 MB)

In [ ]:
# ===== Configuration with ABSOLUTE PATH =====
DATA_URL = "https://files.data.gouv.fr/geo-dvf/latest/csv/2022/full.csv.gz"

# Use absolute path - works from any location
PROJECT_ROOT = Path(r"C:\Users\ahmed\Documents\Projects\real-estate-paris")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FILENAME = "dvf_2022.csv.gz"
FILEPATH = DATA_DIR / FILENAME

print(f"📍 Target location: {FILEPATH}")
print(f"📁 Directory exists: {DATA_DIR.exists()}\n")

# Create directory if not exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Check if file already exists
if FILEPATH.exists():
    file_size_mb = FILEPATH.stat().st_size / (1024 * 1024)
    print(f"✅ File already exists at: {FILEPATH}")
    print(f"📦 Size: {file_size_mb:.2f} MB")
else:
    print(f"⬇️  Downloading from: {DATA_URL}")
    print(f"📁 Saving to: {FILEPATH}")
    print(f"⏰ This may take 5-15 minutes...\n")
    
    response = requests.get(DATA_URL, stream=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    downloaded = 0
    last_printed_mb = 0
    
    with open(FILEPATH, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            mb_downloaded = downloaded / (1024 * 1024)
            if mb_downloaded - last_printed_mb >= 10:
                if total_size > 0:
                    percent = (downloaded / total_size) * 100
                    print(f"  📊 Downloaded: {mb_downloaded:.1f} MB ({percent:.1f}%)")
                else:
                    print(f"  📊 Downloaded: {mb_downloaded:.1f} MB")
                last_printed_mb = mb_downloaded
    
    file_size_mb = FILEPATH.stat().st_size / (1024 * 1024)
    print(f"\n✅ Download complete!")
    print(f"📦 File size: {file_size_mb:.2f} MB")

## 📖 Step 2: Load DVF Dataset into pandas

We load the compressed CSV file directly without extracting it.
pandas can read compressed files automatically.

**Note:** This may take 1-2 minutes due to the file size.

In [ ]:
print("📖 Reading DVF dataset...")
print("⏰ Please wait, this may take 1-2 minutes...\n")

# Read the compressed CSV file
df_dvf = pd.read_csv(
    FILEPATH,
    compression='gzip',
    low_memory=False
)

# Confirmation
print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df_dvf.shape}")
print(f"   • Rows: {df_dvf.shape[0]:,}")
print(f"   • Columns: {df_dvf.shape[1]}")
print(f"💾 Memory usage: {df_dvf.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

## 🔍 Step 3: Explore Data Structure

In [ ]:
# Display first 5 rows
print("🔍 First 5 rows of the dataset:")
df_dvf.head()

In [ ]:
# List all columns
print(f"📋 Total columns: {len(df_dvf.columns)}\n")
print("Column names:")
for i, col in enumerate(df_dvf.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Column information
print("📋 Column information:\n")
df_dvf.info()

In [ ]:
# Statistical summary
print("📊 Statistical summary:")
df_dvf.describe()

## ❓ Step 4: Missing Values Analysis

Missing values are a common problem in real-world datasets.
We need to identify them before any analysis.

According to the project requirements:
- Delete columns with more than 50% missing values
- Delete rows with any missing values (after column cleaning)

In [ ]:
# Count missing values per column
missing_counts = df_dvf.isnull().sum()
missing_percent = (df_dvf.isnull().sum() / len(df_dvf)) * 100

# Combine into a summary DataFrame
missing_summary = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_Percent': missing_percent.values
})

# Sort by percentage descending
missing_summary = missing_summary.sort_values(
    'Missing_Percent', 
    ascending=False
)

# Display
print("❓ Missing Values Summary (sorted by % missing):\n")
missing_summary

In [ ]:
# Quick summary
total_cells = df_dvf.size
missing_cells = df_dvf.isnull().sum().sum()

print(f"📊 Total cells: {total_cells:,}")
print(f"❓ Missing cells: {missing_cells:,}")
print(f"📉 Missing percentage: {(missing_cells/total_cells)*100:.2f}%")
print(f"\n🔍 Columns with missing values:")
print(f"   • >50% missing: {len(missing_summary[missing_summary['Missing_Percent'] > 50])}")
print(f"   • 25-50% missing: {len(missing_summary[(missing_summary['Missing_Percent'] >= 25) & (missing_summary['Missing_Percent'] <= 50)])}")
print(f"   • <25% missing: {len(missing_summary[(missing_summary['Missing_Percent'] > 0) & (missing_summary['Missing_Percent'] < 25)])}")
print(f"   • 0% missing (complete): {len(missing_summary[missing_summary['Missing_Percent'] == 0])}")

In [ ]:
# Filter columns with missing values
cols_with_missing = missing_summary[missing_summary['Missing_Percent'] > 0]

# Plot
plt.figure(figsize=(12, 10))
bars = plt.barh(
    cols_with_missing['Column'], 
    cols_with_missing['Missing_Percent'],
    color=['#e74c3c' if p > 50 else '#f39c12' if p > 25 else '#2ecc71' 
           for p in cols_with_missing['Missing_Percent']]
)
plt.xlabel('Missing Percentage (%)', fontsize=12)
plt.ylabel('Column', fontsize=12)
plt.title('Missing Values per Column (DVF 2022)', fontsize=14, fontweight='bold')
plt.axvline(x=50, color='red', linestyle='--', label='50% threshold (to be removed)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"\n📊 Color legend:")
print(f"  🔴 Red:    >50% missing (will be removed)")
print(f"  🟠 Orange: 25-50% missing")
print(f"  🟢 Green:  <25% missing")

## 🧹 Step 5: Data Cleaning

According to project requirements, we will:
1. **Filter Paris only** (department 75)
2. **Delete columns** with more than 50% missing values
3. **Delete rows** with any remaining missing values

In [ ]:
# ===== Step 5.1: Filter Paris (department 75) =====
print("🇫🇷 Step 5.1: Filtering Paris data only (department 75)\n")

print(f"📊 Before filtering:")
print(f"   • Total rows: {len(df_dvf):,}")
print(f"   • Unique departments: {df_dvf['code_departement'].nunique()}")

# Filter Paris
df_paris = df_dvf[df_dvf['code_departement'] == '75'].copy()

print(f"\n✅ After filtering (Paris only):")
print(f"   • Total rows: {len(df_paris):,}")
print(f"   • Unique departments: {df_paris['code_departement'].nunique()}")
print(f"   • Reduction: {(1 - len(df_paris)/len(df_dvf)) * 100:.2f}%")

print(f"\n🔍 Verification:")
print(f"   • Department codes in filtered data: {df_paris['code_departement'].unique()}")
print(f"\n💾 Memory usage: {df_paris.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

In [ ]:
# ===== Step 5.2: Delete columns with >50% missing values =====
print("🗑️ Step 5.2: Removing columns with >50% missing values\n")

# Calculate missing % for Paris data
missing_paris = (df_paris.isnull().sum() / len(df_paris)) * 100

# Identify columns to drop
cols_to_drop = missing_paris[missing_paris > 50].index.tolist()

print(f"📋 Columns to be removed ({len(cols_to_drop)}):")
for i, col in enumerate(cols_to_drop, 1):
    print(f"   {i:2d}. {col} ({missing_paris[col]:.2f}% missing)")

# Drop them
df_paris_cleaned = df_paris.drop(columns=cols_to_drop)

print(f"\n✅ Result:")
print(f"   • Before: {len(df_paris.columns)} columns")
print(f"   • After: {len(df_paris_cleaned.columns)} columns")
print(f"   • Removed: {len(cols_to_drop)} columns")

print(f"\n📋 Remaining columns ({len(df_paris_cleaned.columns)}):")
for i, col in enumerate(df_paris_cleaned.columns, 1):
    pct = (df_paris_cleaned[col].isnull().sum() / len(df_paris_cleaned)) * 100
    status = "✅" if pct == 0 else "⚠️" if pct < 25 else "🟠"
    print(f"   {status} {i:2d}. {col} ({pct:.2f}% missing)")

In [ ]:
# ===== Step 5.3: Delete rows with any missing values =====
print("🗑️ Step 5.3: Removing rows with any missing values\n")

print(f"📊 Before dropping rows:")
print(f"   • Total rows: {len(df_paris_cleaned):,}")
print(f"   • Total missing cells: {df_paris_cleaned.isnull().sum().sum():,}")

# Drop rows with any NaN
df_paris_final = df_paris_cleaned.dropna()

print(f"\n✅ After dropping rows:")
print(f"   • Total rows: {len(df_paris_final):,}")
print(f"   • Total missing cells: {df_paris_final.isnull().sum().sum():,}")
print(f"   • Reduction: {(1 - len(df_paris_final)/len(df_paris_cleaned)) * 100:.2f}%")

print(f"\n🎯 Final Clean Dataset:")
print(f"   • Rows: {len(df_paris_final):,}")
print(f"   • Columns: {len(df_paris_final.columns)}")
print(f"   • Missing values: {df_paris_final.isnull().sum().sum():,}")
print(f"   • Memory usage: {df_paris_final.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

## 💾 Step 6: Save Cleaned Data

We save the cleaned Paris dataset to `data/processed/` for use 
in the next steps of the project (Database Modeling, ETL).

In [ ]:
# ===== Step 6: Save cleaned data =====
print("💾 Saving cleaned Paris dataset...\n")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / "dvf_paris_2022_cleaned.csv"

# Save as CSV
df_paris_final.to_csv(OUTPUT_FILE, index=False)

# Verify
file_size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
print(f"✅ File saved successfully!")
print(f"📍 Location: {OUTPUT_FILE}")
print(f"📦 File size: {file_size_mb:.2f} MB")
print(f"📊 Rows: {len(df_paris_final):,}")
print(f"📋 Columns: {len(df_paris_final.columns)}")

# Save compressed version
COMPRESSED_FILE = PROCESSED_DIR / "dvf_paris_2022_cleaned.csv.gz"
df_paris_final.to_csv(COMPRESSED_FILE, index=False, compression='gzip')

compressed_size_mb = COMPRESSED_FILE.stat().st_size / (1024 * 1024)
print(f"\n📦 Compressed version:")
print(f"   • Location: {COMPRESSED_FILE}")
print(f"   • Size: {compressed_size_mb:.2f} MB")
print(f"   • Compression ratio: {(1 - compressed_size_mb/file_size_mb)*100:.1f}%")

## 📝 Summary

### 🎯 What We Accomplished

**Starting point:**
- 📦 Raw DVF 2022 dataset from data.gouv.fr
- 📊 4,676,187 rows × 40 columns
- 💾 5 GB in memory

**Cleaning steps applied:**
1. ✅ Filtered to Paris only (department 75)
2. ✅ Removed 19 columns with >50% missing values
3. ✅ Removed rows with any remaining missing values

**Final clean dataset:**
- 📊 45,207 rows × 21 columns
- 💾 31.73 MB in memory
- ✨ 0 missing values

### 📊 Key Findings

1. **Data Quality Issues:**
   - 48.91% of all cells had missing values
   - 19 columns (almost half) had >50% missing data
   - Some columns were 100% empty for Paris data

2. **Paris Real Estate Activity:**
   - 98,308 transactions in Paris in 2022 (before cleaning)
   - 45,207 valid transactions with complete data

### 🔍 Notable Observations

- `surface_reelle_bati` had 48.83% missing — just below the 50% threshold
- This single column caused ~54% row loss when removing incomplete rows
- **Recommendation:** Consider alternative imputation strategies for important columns

### 🚀 Next Steps

1. **Connect to opendata.paris API** for additional data:
   - Risk sectors
   - Rent control prices
   - Green spaces
2. **Design database schema** (3NF) for Step 2
3. **Build ETL pipeline** for Step 3
4. **Create Power BI dashboard** for Step 4